# Feature Engineering cho Ngày Lễ (Holiday Feature Engineering)

Notebook này tạo ra các feature liên quan đến ngày lễ từ `holidays_events.csv` kết hợp với `stores.csv`, nhằm áp dụng đúng phạm vi địa lý (National / Regional / Local) cho từng cặp store-date.

## Các Feature được tạo ra
| Feature | Mô tả |
|---|---|
| `is_national_holiday` | 1 nếu ngày đó là ngày lễ quốc gia |
| `is_regional_holiday` | 1 nếu là ngày lễ vùng, khớp với state của store |
| `is_local_holiday` | 1 nếu là ngày lễ địa phương, khớp với city của store |
| `is_transferred_holiday` | 1 nếu ngày lễ đó ban đầu bị chuyển sang ngày khác |
| `holiday_type` | Mã số (numeric) của loại ngày lễ |
| `is_carnaval_feature` | 1 nếu ngày lễ đó là Carnaval |
| `days_to_next_holiday` | Số ngày cho đến ngày lễ tiếp theo |
| `days_after_last_holiday` | Số ngày kể từ ngày lễ gần nhất |
| `is_earthquake_period` | 1 trong tháng sau trận động đất Ecuador tháng 4/2016 |

## Tổng quan Pipeline
```
Load Data → Preprocess Dates → Filter Transferred Holidays
    → Encode Types → Flag Carnaval → Merge Geographic Scope
    → Apply Holiday Flags → Halo Effect → Earthquake Dummy → Output
```

## 1. Import Thư viện

Sử dụng **pandas** cho toàn bộ thao tác dữ liệu và **numpy** cho các phép tính số học.

In [3]:
import pandas as pd
import numpy as np

## 2. Đọc Dữ liệu (Load Data)

Pipeline cần ba bộ dữ liệu:
- **`train.csv`** — lịch sử giao dịch/doanh số chính (cần có cột `date` và `store_nbr`).
- **`holidays_events_cleaned.csv`** — lịch ngày lễ và sự kiện đã được làm sạch.
- **`stores_cleaned.csv`** — thông tin store, chứa `city` và `state` để khớp địa lý.

In [4]:
BASE_PATH = r'D:\Topic_13_Project\Topic_13_Retail_Store_Sales_Time_Series\data'

df_train    = pd.read_csv(rf'{BASE_PATH}\processed\train_cleaned.csv')
df_holidays = pd.read_csv(rf'{BASE_PATH}\processed\holidays_events_cleaned.csv')
df_stores   = pd.read_csv(rf'{BASE_PATH}\processed\stores_cleaned.csv')

print(f'Train shape      : {df_train.shape}')
print(f'Holidays shape   : {df_holidays.shape}')
print(f'Stores shape     : {df_stores.shape}')
df_train.head(3)

Train shape      : (3000888, 6)
Holidays shape   : (350, 6)
Stores shape     : (54, 6)


,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0
1,1,2013-01-01,1,BABY CARE,0.0,0
2,2,2013-01-01,1,BEAUTY,0.0,0


## 3. Bước 1 — Tiền xử lý Ngày tháng (Date Preprocessing)

Chuyển đổi cột `date` sang kiểu `datetime` trong cả `main_df` và `holidays_events_df` để có thể thực hiện các phép so sánh và tính toán trên ngày tháng.

In [5]:
df_train['date']    = pd.to_datetime(df_train['date'])
df_holidays['date'] = pd.to_datetime(df_holidays['date'])

print('Date columns converted.')
print(f"Train date range   : {df_train['date'].min()} → {df_train['date'].max()}")
print(f"Holiday date range : {df_holidays['date'].min()} → {df_holidays['date'].max()}")

Date columns converted.
Train date range   : 2013-01-01 00:00:00 → 2017-08-15 00:00:00
Holiday date range : 2012-03-02 00:00:00 → 2017-12-26 00:00:00


## 4. Bước 2 — Xử lý Transferred Holidays

Trong hệ thống ngày lễ của Ecuador, một ngày lễ có thể bị **chuyển (transferred)**:
- Dòng có `transferred = True` nghĩa là ngày gốc đó **bị hủy** (nhân viên vẫn đi làm bình thường).
- Ngày nghỉ thực tế được bù sang một dòng khác với `type = 'Transfer'`.

Ta **chỉ giữ lại các dòng có `transferred == False`** để mô hình hóa đúng các ngày nghỉ thực tế.
Ngoài ra, tạo flag numeric `is_transferred` để ghi nhận rằng ngày lễ đó ban đầu đã bị dời lịch.

In [6]:
df_holidays['is_transferred'] = df_holidays['transferred'].astype(int)
valid_holidays = df_holidays[df_holidays['transferred'] == False].copy()

print(f'Total holiday rows       : {len(df_holidays)}')
print(f'Valid (non-transferred)  : {len(valid_holidays)}')
print(f'Removed (transferred)    : {len(df_holidays) - len(valid_holidays)}')
valid_holidays[['date', 'type', 'locale', 'locale_name', 'transferred']].head()

Total holiday rows       : 350
Valid (non-transferred)  : 338
Removed (transferred)    : 12


,date,type,locale,locale_name,transferred
0,2012-03-02,Holiday,Local,Manta,False
1,2012-04-01,Holiday,Regional,Cotopaxi,False
2,2012-04-12,Holiday,Local,Cuenca,False
3,2012-04-14,Holiday,Local,Libertad,False
4,2012-04-21,Holiday,Local,Riobamba,False


## 5. Bước 3 — Mã hóa Holiday Type sang Số (Encode to Numeric)

Cột `type` đang là dạng string. Ta map sang số nguyên có thứ tự để các mô hình tree-based có thể học được mối quan hệ giữa các loại ngày lễ.

| Type | Code | Ý nghĩa |
|---|---|---|
| Work Day | 0 | Không phải ngày lễ — nhân viên đi làm bình thường |
| Holiday | 1 | Ngày lễ quốc gia tiêu chuẩn |
| Event | 2 | Sự kiện đặc biệt (không phải luật định) |
| Additional | 3 | Ngày lễ bổ sung được ban hành bởi sắc lệnh |
| Transfer | 4 | Ngày nghỉ bù từ một transferred holiday |
| Bridge | 5 | Ngày cầu nối giữa ngày lễ và cuối tuần |

In [7]:
type_mapping = {
    'Holiday'   : 1,
    'Event'     : 2,
    'Additional': 3,
    'Transfer'  : 4,
    'Bridge'    : 5,
    'Work Day'  : 0
}

valid_holidays['holiday_type_encoded'] = valid_holidays['type'].map(type_mapping).fillna(0)

print('Holiday type distribution after encoding:')
valid_holidays.groupby(['type', 'holiday_type_encoded']).size().reset_index(name='count')

Holiday type distribution after encoding:


,type,holiday_type_encoded,count
0,Additional,3,51
1,Bridge,5,5
2,Event,2,56
3,Holiday,1,209
4,Transfer,4,12
5,Work Day,0,5


## 6. Bước 4 — Tạo Feature Carnaval

**Carnaval** là lễ hội lớn kéo dài nhiều ngày tại Ecuador, có tác động rõ rệt và khác biệt đến doanh số bán lẻ (đám đông lớn, du lịch, các sản phẩm đặc thù tăng đột biến).  
Ta tách riêng một binary flag bằng cách tìm kiếm từ `'Carnaval'` trong cột `description` (không phân biệt hoa thường).

In [8]:
valid_holidays['is_carnaval'] = (
    valid_holidays['description']
    .str.contains('Carnaval', case=False, na=False)
    .astype(int)
)

carnaval_dates = valid_holidays[valid_holidays['is_carnaval'] == 1][['date', 'description', 'locale']]
print(f'Carnaval event rows found: {len(carnaval_dates)}')
carnaval_dates

Carnaval event rows found: 10


,date,description,locale
44,2013-02-11,Carnaval,National
45,2013-02-12,Carnaval,National
94,2014-03-03,Carnaval,National
95,2014-03-04,Carnaval,National
162,2015-02-16,Carnaval,National
163,2015-02-17,Carnaval,National
212,2016-02-08,Carnaval,National
213,2016-02-09,Carnaval,National
299,2017-02-27,Carnaval,National
300,2017-02-28,Carnaval,National


## 7. Bước 5 — Merge Phạm vi Địa lý (Geographic Scope)

Ngày lễ ở Ecuador có ba phạm vi áp dụng:
- **National** — áp dụng cho toàn bộ store trên cả nước.
- **Regional** — chỉ áp dụng cho các store thuộc một *state* cụ thể (`locale_name` = tên state).
- **Local** — chỉ áp dụng cho các store thuộc một *city* cụ thể (`locale_name` = tên city).

Ta join dataframe chính với `stores_df` để gắn `city` và `state` vào mỗi dòng store-date, phục vụ cho bước lọc địa lý ở bước tiếp theo.

In [9]:
df = df_train.merge(
    df_stores[['store_nbr', 'city', 'state']],
    on='store_nbr',
    how='left'
)

print(f'Merged dataframe shape : {df.shape}')
print(f'Unique stores          : {df["store_nbr"].nunique()}')
print(f'States in data         : {sorted(df["state"].unique())}')
df[['date', 'store_nbr', 'city', 'state']].drop_duplicates('store_nbr').head(5)

Merged dataframe shape : (3000888, 8)
Unique stores          : 54
States in data         : ['Azuay', 'Bolivar', 'Chimborazo', 'Cotopaxi', 'El Oro', 'Esmeraldas', 'Guayas', 'Imbabura', 'Loja', 'Los Rios', 'Manabi', 'Pastaza', 'Pichincha', 'Santa Elena', 'Santo Domingo de los Tsachilas', 'Tungurahua']


,date,store_nbr,city,state
0,2013-01-01,1,Quito,Pichincha
33,2013-01-01,10,Quito,Pichincha
66,2013-01-01,11,Cayambe,Pichincha
99,2013-01-01,12,Latacunga,Cotopaxi
132,2013-01-01,13,Latacunga,Cotopaxi


## 8. Bước 6 — Xây dựng và Áp dụng Holiday Flags

### 8.1 Tính toán trước các Mask

Với mỗi dòng ngày lễ hợp lệ, ta tính một boolean mask trên `df` dựa vào locale:
- **National** → tất cả các dòng có `date == holiday_date`.
- **Regional** → các dòng có `date == holiday_date` **VÀ** `state == locale_name`.
- **Local** → các dòng có `date == holiday_date` **VÀ** `city == locale_name`.

Ta thu thập tất cả mask trước để tránh lặp nhiều lần trên toàn bộ dataframe.

In [10]:
holiday_features = []

for _, row in valid_holidays.iterrows():
    h_date       = row['date']
    h_locale     = row['locale']
    h_locale_name = row['locale_name']
    h_encoded    = row['holiday_type_encoded']
    is_transferred = row['is_transferred']
    is_carnaval  = row['is_carnaval']

    if h_locale == 'National':
        mask = (df['date'] == h_date)
    elif h_locale == 'Regional':
        mask = (df['date'] == h_date) & (df['state'] == h_locale_name)
    elif h_locale == 'Local':
        mask = (df['date'] == h_date) & (df['city'] == h_locale_name)
    else:
        continue

    holiday_features.append({
        'mask'         : mask,
        'locale'       : h_locale,
        'is_transferred': is_transferred,
        'encoded_type' : h_encoded,
        'is_carnaval'  : is_carnaval
    })

print(f'Total holiday mask entries built: {len(holiday_features)}')

Total holiday mask entries built: 338


### 8.2 Khởi tạo các cột Holiday

Tất cả các cột flag được khởi tạo bằng **0** (không phải ngày lễ) trước khi ta gán giá trị 1 cho các dòng thỏa điều kiện.

In [11]:
df['is_national_holiday']  = 0
df['is_regional_holiday']  = 0
df['is_local_holiday']     = 0
df['is_transferred_holiday'] = 0
df['holiday_type']         = 0
df['is_carnaval_feature']  = 0

print('Holiday columns initialised to 0:')
print(df[['is_national_holiday', 'is_regional_holiday', 'is_local_holiday',
          'is_transferred_holiday', 'holiday_type', 'is_carnaval_feature']].sum())

Holiday columns initialised to 0:
is_national_holiday       0
is_regional_holiday       0
is_local_holiday          0
is_transferred_holiday    0
holiday_type              0
is_carnaval_feature       0
dtype: int64


### 8.3 Gán Giá trị Flag

Ta duyệt qua danh sách mask đã tính và dùng `.loc[]` để gán giá trị flag tương ứng.  
Nếu nhiều ngày lễ rơi vào cùng một ngày (ví dụ vừa có National vừa có Local event), lần gán cuối cùng sẽ thắng đối với `holiday_type` và `is_carnaval_feature`, nhưng các scope flag được gán độc lập với nhau.

In [12]:
for h in holiday_features:
    if h['locale'] == 'National':
        df.loc[h['mask'], 'is_national_holiday'] = 1
    elif h['locale'] == 'Regional':
        df.loc[h['mask'], 'is_regional_holiday'] = 1
    elif h['locale'] == 'Local':
        df.loc[h['mask'], 'is_local_holiday'] = 1

    df.loc[h['mask'], 'is_transferred_holiday'] = h['is_transferred']
    df.loc[h['mask'], 'holiday_type']           = h['encoded_type']
    df.loc[h['mask'], 'is_carnaval_feature']    = h['is_carnaval']

print('Holiday flags assigned. Summary:')
print(df[['is_national_holiday', 'is_regional_holiday', 'is_local_holiday',
          'is_transferred_holiday', 'is_carnaval_feature']].sum())

Holiday flags assigned. Summary:
is_national_holiday       242352
is_regional_holiday         1023
is_local_holiday           11880
is_transferred_holiday         0
is_carnaval_feature        17820
dtype: int64


## 9. Bước 7 — Halo Effect Features (Hiệu ứng Lan tỏa)

Hành vi mua sắm thay đổi trong **những ngày trước** và **những ngày sau** ngày lễ ("halo effect" — hiệu ứng lan tỏa):
- **Pre-holiday surge**: người mua sắm tích trữ hàng 1–3 ngày trước ngày lễ.
- **Post-holiday dip**: nhu cầu giảm ngay sau kỳ nghỉ.

Ta tính hai feature:
- `days_to_next_holiday` — số ngày cho đến ngày lễ tiếp theo (bất kể locale).
- `days_after_last_holiday` — số ngày đã qua kể từ ngày lễ gần nhất.

> **Tối ưu hóa**: Tính các giá trị này trên tập **ngày duy nhất** trong `df` rồi left-merge lại, tránh tính lặp cho từng dòng store cùng ngày.

In [13]:
unique_holiday_dates = (
    valid_holidays['date']
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

def get_days_to_next(date):
    """Return the number of days until the next holiday, or -1 if none exists."""
    future = unique_holiday_dates[unique_holiday_dates > date]
    return (future.iloc[0] - date).days if not future.empty else -1

def get_days_after_last(date):
    """Return the number of days since the last holiday, or -1 if none exists."""
    past = unique_holiday_dates[unique_holiday_dates < date]
    return (date - past.iloc[-1]).days if not past.empty else -1

# Compute on unique dates only (much faster)
unique_dates = df[['date']].drop_duplicates().copy()
unique_dates['days_to_next_holiday']   = unique_dates['date'].apply(get_days_to_next)
unique_dates['days_after_last_holiday'] = unique_dates['date'].apply(get_days_after_last)

# Merge back to the main dataframe
df = df.merge(unique_dates, on='date', how='left')

print('Halo effect features added:')
print(df[['days_to_next_holiday', 'days_after_last_holiday']].describe())

Halo effect features added:
       days_to_next_holiday  days_after_last_holiday
count          3.000888e+06             3.000888e+06
mean           1.072862e+01             1.072387e+01
std            1.052096e+01             1.052355e+01
min            1.000000e+00             1.000000e+00
25%            3.000000e+00             3.000000e+00
50%            7.000000e+00             7.000000e+00
75%            1.600000e+01             1.600000e+01
max            6.000000e+01             6.000000e+01


## 10. Bước 8 — Earthquake Dummy (Tháng 4 – 5/2016)

Một trận **động đất 7.8 độ Richter** xảy ra tại Ecuador vào **ngày 16/04/2016**, gây gián đoạn nghiêm trọng chuỗi cung ứng và hành vi tiêu dùng.  
Tác động lên doanh số bán lẻ có thể quan sát rõ trong khoảng **một tháng** sau sự kiện.

Ta nắm bắt đứt gãy cấu trúc này bằng một biến dummy nhị phân:
- `is_earthquake_period = 1` cho các ngày trong khoảng `[2016-04-16, 2016-05-16]`.
- `is_earthquake_period = 0` cho tất cả ngày còn lại.

In [14]:
earthquake_start = pd.to_datetime('2016-04-16')
earthquake_end   = pd.to_datetime('2016-05-16')

df['is_earthquake_period'] = (
    (df['date'] >= earthquake_start) & (df['date'] <= earthquake_end)
).astype(int)

n_earthquake_rows = df['is_earthquake_period'].sum()
print(f'Rows flagged as earthquake period : {n_earthquake_rows}')
print(f'Unique dates in earthquake period  : {df[df["is_earthquake_period"]==1]["date"].nunique()}')

Rows flagged as earthquake period : 55242
Unique dates in earthquake period  : 31


## 11. Dọn dẹp — Xóa các cột tạm (Drop Temporary Columns)

Các cột `city` và `state` chỉ được thêm vào để phục vụ việc khớp địa lý ở Bước 5–6.  
Ta xóa chúng đi để dataframe đầu ra gọn gàng, chỉ chứa các feature đã được tạo ra.

In [15]:
df = df.drop(columns=['city', 'state'])

print(f'Final dataframe shape : {df.shape}')
print('\nColumns:')
print(df.columns.tolist())

Final dataframe shape : (3000888, 15)

Columns:
['id', 'date', 'store_nbr', 'family', 'sales', 'onpromotion', 'is_national_holiday', 'is_regional_holiday', 'is_local_holiday', 'is_transferred_holiday', 'holiday_type', 'is_carnaval_feature', 'days_to_next_holiday', 'days_after_last_holiday', 'is_earthquake_period']


## 12. Kiểm tra Đầu ra (Output Preview & Summary)

Kiểm tra lần cuối các feature đã được tạo — thống kê tổng quan và một vài dòng mẫu để xác nhận kết quả đúng.

In [16]:
holiday_cols = [
    'is_national_holiday', 'is_regional_holiday', 'is_local_holiday',
    'is_transferred_holiday', 'holiday_type', 'is_carnaval_feature',
    'days_to_next_holiday', 'days_after_last_holiday', 'is_earthquake_period'
]

print('=== Feature Summary ===')
print(df[holiday_cols].describe().T[['mean', 'min', 'max']])
print()
print('=== Sample rows with active holidays ===')
df[df['is_national_holiday'] == 1][['date', 'store_nbr'] + holiday_cols].head(10)

=== Feature Summary ===
                              mean  min   max
is_national_holiday       0.080760  0.0   1.0
is_regional_holiday       0.000341  0.0   1.0
is_local_holiday          0.003959  0.0   1.0
is_transferred_holiday    0.000000  0.0   0.0
holiday_type              0.167645  0.0   5.0
is_carnaval_feature       0.005938  0.0   1.0
days_to_next_holiday     10.728622  1.0  60.0
days_after_last_holiday  10.723872  1.0  60.0
is_earthquake_period      0.018409  0.0   1.0

=== Sample rows with active holidays ===


,date,store_nbr,is_national_holiday,is_regional_holiday,is_local_holiday,is_transferred_holiday,holiday_type,is_carnaval_feature,days_to_next_holiday,days_after_last_holiday,is_earthquake_period
0,2013-01-01,1,1,0,0,0,1,0,4,1,0
1,2013-01-01,1,1,0,0,0,1,0,4,1,0
2,2013-01-01,1,1,0,0,0,1,0,4,1,0
3,2013-01-01,1,1,0,0,0,1,0,4,1,0
4,2013-01-01,1,1,0,0,0,1,0,4,1,0
5,2013-01-01,1,1,0,0,0,1,0,4,1,0
6,2013-01-01,1,1,0,0,0,1,0,4,1,0
7,2013-01-01,1,1,0,0,0,1,0,4,1,0
8,2013-01-01,1,1,0,0,0,1,0,4,1,0
9,2013-01-01,1,1,0,0,0,1,0,4,1,0


## 13. Lưu Kết quả (Save Output)

Lưu dataframe đã được bổ sung feature vào thư mục processed để các notebook modelling downstream có thể sử dụng.

In [17]:
OUTPUT_PATH = rf'{BASE_PATH}\processed\train_holiday_features.csv'
df.to_csv(OUTPUT_PATH, index=False)
print(f'Saved to: {OUTPUT_PATH}')
print(f'Shape    : {df.shape}')

Saved to: D:\Topic_13_Project\Topic_13_Retail_Store_Sales_Time_Series\data\processed\train_holiday_features.csv
Shape    : (3000888, 15)
